# DATA PROCESSING — MovieLens ml-25m

Full data processing pipeline for **MovieLens ml-25m** (25M ratings, ~62K movies, ~162K users).  
Steps:

1. **Review Dataset** — Inspect 6 CSV files from ml-25m  
2. **Data Cleaning** — Handle nulls, duplicates, data types  
3. **TMDB API Enrichment** — Fetch metadata (budget, revenue, runtime, vote_average) + compute `imdb_score`  
4. **Data Reduction** — Feature selection  
5. **Data Transformation** — OneHot genres, TF-IDF tags, Genome Tags, Normalization  
6. **Data Discretization** — Bin release year and runtime

---
**Requirement**: Place the `ml-25m/` folder inside `data/origin/` before running.  
Tải tại: https://grouplens.org/datasets/movielens/25m/


## I. Review Dataset

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import re
import os
import json
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# -- Paths ------------------------------------------------------------------
DATA_ORIGIN  = Path('data/origin/ml-25m')
DATA_CLEAN   = Path('data/cleaning')
CACHE_FILE   = DATA_CLEAN / 'tmdb_cache.json'
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

# -- TMDb API ---------------------------------------------------------------
TMDB_API_KEY = '4d3a1be95ec76ac97e468425e4f746a9'   # Replace with your API key
TMDB_BASE    = 'https://api.themoviedb.org/3'

print('Python path setup OK')
print('DATA_ORIGIN:', DATA_ORIGIN.resolve())
print('DATA_CLEAN :', DATA_CLEAN.resolve())

Python path setup OK
DATA_ORIGIN: E:\Data Mining\recommendation-movies-system\data\origin\ml-25m
DATA_CLEAN : E:\Data Mining\recommendation-movies-system\data\cleaning


In [2]:
def inspect_data(df, name='Dataset'):
    """Inspect a DataFrame comprehensively."""
    sep = '=' * 50
    print(sep)
    print(f'  DATASET: {name.upper()}')
    print(sep)
    print(f'  Shape: {df.shape[0]:,} rows x {df.shape[1]} cols')
    print('\n1. DTYPES:')
    print(df.dtypes)
    null_info = df.isnull().sum()
    null_info = null_info[null_info > 0]
    if len(null_info):
        print('\n2. NULL VALUES:')
        print(null_info)
    else:
        print('\n2. NULL VALUES: None')
    dup = df.duplicated().sum()
    print(f'\n3. DUPLICATES: {dup:,}')
    print('\n4. SAMPLE (5 rows):')
    display(df.sample(min(5, len(df))))

### Load 6 CSV Files from ml-25m

ml-25m contains: `ratings.csv`, `movies.csv`, `tags.csv`, `links.csv`, `genome-scores.csv`, `genome-tags.csv`

In [3]:
# Check that required files exist
required_files = [
    'ratings.csv', 'movies.csv', 'tags.csv',
    'links.csv', 'genome-scores.csv', 'genome-tags.csv'
]
for f in required_files:
    path = DATA_ORIGIN / f
    exists = path.exists()
    size   = f'{path.stat().st_size / 1e6:.1f} MB' if exists else 'MISSING'
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {f}: {size}')

[OK] ratings.csv: 678.3 MB
[OK] movies.csv: 3.0 MB
[OK] tags.csv: 38.8 MB
[OK] links.csv: 1.4 MB
[OK] genome-scores.csv: 435.2 MB
[OK] genome-tags.csv: 0.0 MB


In [4]:
print('Loading ratings.csv (25M rows)...')
df_ratings = pd.read_csv(DATA_ORIGIN / 'ratings.csv',
                         dtype={'userId': 'int32', 'movieId': 'int32',
                                'rating': 'float32'})
inspect_data(df_ratings, 'Ratings')

Loading ratings.csv (25M rows)...
  DATASET: RATINGS
  Shape: 25,000,095 rows x 4 cols

1. DTYPES:
userId         int32
movieId        int32
rating       float32
timestamp      int64
dtype: object

2. NULL VALUES: None

3. DUPLICATES: 0

4. SAMPLE (5 rows):


,userId,movieId,rating,timestamp
6376596,41382,762,2.0,1107212087
20221547,131444,271,2.0,996520665
13180278,85318,899,0.5,1415984252
22087001,143578,3873,3.0,974329119
5404572,35069,1409,3.0,941557540


In [5]:
df_movies = pd.read_csv(DATA_ORIGIN / 'movies.csv')
inspect_data(df_movies, 'Movies')

  DATASET: MOVIES
  Shape: 62,423 rows x 3 cols

1. DTYPES:
movieId    int64
title        str
genres       str
dtype: object

2. NULL VALUES: None

3. DUPLICATES: 0

4. SAMPLE (5 rows):


,movieId,title,genres
25050,122876,Torrente 4: Lethal Crisis (2011),Action|Comedy|Crime
42722,164481,The Trap (1966),Adventure|Drama|Romance
463,468,Englishman Who Went Up a Hill But Came Down a ...,Comedy|Romance
29914,135005,Vedo nudo (1969),(no genres listed)
50908,182023,Canine Caddy (1941),Animation


In [6]:
print('Loading tags.csv...')
df_tags = pd.read_csv(DATA_ORIGIN / 'tags.csv')
inspect_data(df_tags, 'Tags')

Loading tags.csv...
  DATASET: TAGS
  Shape: 1,093,360 rows x 4 cols

1. DTYPES:
userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object

2. NULL VALUES:
tag    16
dtype: int64

3. DUPLICATES: 0

4. SAMPLE (5 rows):


,userId,movieId,tag,timestamp
504587,55164,124394,deborah kerr,1427490507
596768,68238,296,multiple storylines,1435699917
263495,14526,2951,Ennio Morricone,1571741532
837777,116587,7254,don't remember,1212249743
232899,8705,69481,Ralph Fiennes,1297683359


In [7]:
df_links = pd.read_csv(DATA_ORIGIN / 'links.csv')
inspect_data(df_links, 'Links')

  DATASET: LINKS
  Shape: 62,423 rows x 3 cols

1. DTYPES:
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object

2. NULL VALUES:
tmdbId    107
dtype: int64

3. DUPLICATES: 0

4. SAMPLE (5 rows):


,movieId,imdbId,tmdbId
21289,109864,2771372,177494.0
28144,131019,3496372,322518.0
20604,106491,1335975,64686.0
2916,3008,156729,16129.0
39467,157260,98046,335807.0


In [8]:
print('Loading genome-scores.csv (~15M rows)...')
df_genome_scores = pd.read_csv(DATA_ORIGIN / 'genome-scores.csv',
                                dtype={'movieId': 'int32', 'tagId': 'int32',
                                       'relevance': 'float32'})
df_genome_tags   = pd.read_csv(DATA_ORIGIN / 'genome-tags.csv')
inspect_data(df_genome_scores, 'Genome Scores (sample)')
inspect_data(df_genome_tags,   'Genome Tags')

Loading genome-scores.csv (~15M rows)...
  DATASET: GENOME SCORES (SAMPLE)
  Shape: 15,584,448 rows x 3 cols

1. DTYPES:
movieId        int32
tagId          int32
relevance    float32
dtype: object

2. NULL VALUES: None

3. DUPLICATES: 0

4. SAMPLE (5 rows):


,movieId,tagId,relevance
1690062,1679,319,0.03300
12564241,96821,578,0.02675
2843595,2807,1036,0.05050
7212823,7320,392,0.02275
12282562,91831,899,0.08250


  DATASET: GENOME TAGS
  Shape: 1,128 rows x 2 cols

1. DTYPES:
tagId    int64
tag        str
dtype: object

2. NULL VALUES: None

3. DUPLICATES: 0

4. SAMPLE (5 rows):


,tagId,tag
987,988,super-hero
46,47,almodovar
1126,1127,zombie
767,768,parenthood
682,683,murder mystery


## II. Data Cleaning

In [9]:
# -- Ratings ----------------------------------------------------------------
print('=== RATINGS CLEANING ===')
# Remove duplicates
before = len(df_ratings)
df_ratings.drop_duplicates(subset=['userId', 'movieId'], keep='last', inplace=True)
print(f'Dropped {before - len(df_ratings):,} duplicate ratings')

# Filter users with too few ratings (< 5) — heavy cold-start
user_counts = df_ratings['userId'].value_counts()
valid_users = user_counts[user_counts >= 5].index
df_ratings  = df_ratings[df_ratings['userId'].isin(valid_users)].reset_index(drop=True)
print(f'Kept {df_ratings["userId"].nunique():,} users (>= 5 ratings each)')
print(f'Total ratings: {len(df_ratings):,}')

# Filter movies with too few ratings (< 5)
movie_counts  = df_ratings['movieId'].value_counts()
valid_movies  = movie_counts[movie_counts >= 5].index
df_ratings    = df_ratings[df_ratings['movieId'].isin(valid_movies)].reset_index(drop=True)
print(f'Kept {df_ratings["movieId"].nunique():,} movies (>= 5 ratings each)')
print(f'Final ratings: {len(df_ratings):,}')

=== RATINGS CLEANING ===
Dropped 0 duplicate ratings
Kept 162,541 users (>= 5 ratings each)
Total ratings: 25,000,095
Kept 32,720 movies (>= 5 ratings each)
Final ratings: 24,945,870


In [10]:
# -- Movies -----------------------------------------------------------------
print('=== MOVIES CLEANING ===')
df_movies.drop_duplicates(subset='movieId', inplace=True)
# Filter movies present in ratings
df_movies = df_movies[df_movies['movieId'].isin(valid_movies)].reset_index(drop=True)
print(f'Movies after filter: {len(df_movies):,}')

# -- Tags -------------------------------------------------------------------
print('\n=== TAGS CLEANING ===')
df_tags.drop_duplicates(inplace=True)
df_tags.dropna(subset=['tag'], inplace=True)
df_tags = df_tags[df_tags['movieId'].isin(valid_movies)]
# Aggregate tags per movie
df_tags_agg = (df_tags.groupby('movieId')['tag']
               .apply(lambda x: ' '.join(x.str.lower().str.strip().unique()))
               .reset_index())
print(f'Movies with tags: {len(df_tags_agg):,}')

# -- Links ------------------------------------------------------------------
print('\n=== LINKS CLEANING ===')
df_links = df_links[df_links['movieId'].isin(valid_movies)].copy()
df_links['tmdbId'] = pd.to_numeric(df_links['tmdbId'], errors='coerce').astype('Int64')
df_links['imdbId'] = pd.to_numeric(df_links['imdbId'], errors='coerce').astype('Int64')
missing_tmdb = df_links['tmdbId'].isna().sum()
print(f'Missing tmdbId: {missing_tmdb:,}')

=== MOVIES CLEANING ===
Movies after filter: 32,720

=== TAGS CLEANING ===
Movies with tags: 28,785

=== LINKS CLEANING ===
Missing tmdbId: 36


In [11]:
# -- Fix missing tmdbId from imdbId via TMDb API ----------------------------
def get_tmdb_id_from_imdb(imdb_id, api_key=TMDB_API_KEY):
    try:
        fmt_id = f'tt{int(imdb_id):07d}'
        url    = f'{TMDB_BASE}/find/{fmt_id}'
        r      = requests.get(url, params={'api_key': api_key, 'external_source': 'imdb_id'}, timeout=8)
        if r.status_code == 200:
            results = r.json().get('movie_results', [])
            return results[0]['id'] if results else None
    except Exception:
        pass
    return None

missing_mask = df_links['tmdbId'].isna() & df_links['imdbId'].notna()
missing_rows = df_links[missing_mask].copy()
print(f'Attempting to fix {len(missing_rows):,} missing tmdbIds via API...')

for idx, row in tqdm(missing_rows.iterrows(), total=len(missing_rows)):
    tmdb_id = get_tmdb_id_from_imdb(row['imdbId'])
    if tmdb_id:
        df_links.at[idx, 'tmdbId'] = int(tmdb_id)
    time.sleep(0.25)  # rate limit: 4 req/s

still_missing = df_links['tmdbId'].isna().sum()
print(f'Still missing after API fix: {still_missing:,}')

Attempting to fix 36 missing tmdbIds via API...


  0%|          | 0/36 [00:00<?, ?it/s]

Still missing after API fix: 20


## III. TMDB API Enrichment + IMDb Score

Call TMDb API to fetch: `budget`, `revenue`, `runtime`, `vote_average`, `vote_count`, `popularity`.  
Then compute **IMDb Score** using **Bayesian Weighted Rating** (IMDb formula):

$$\text{imdb\_score} = \frac{v}{v+m} \times R + \frac{m}{v+m} \times C$$

Where: v = vote_count, R = vote_average, C = global mean, m = 60th percentile

**Caching**: Results are stored in `data/cleaning/tmdb_cache.json` so the notebook can resume without re-fetching.

In [12]:
# Load cache if available
if CACHE_FILE.exists():
    with open(CACHE_FILE, 'r') as f:
        tmdb_cache = json.load(f)
    print(f'Loaded {len(tmdb_cache):,} entries from cache')
else:
    tmdb_cache = {}
    print('No cache found, starting fresh')

def fetch_tmdb_metadata(tmdb_id, api_key=TMDB_API_KEY):
    """Fetch metadata from TMDb for a single movie."""
    key = str(tmdb_id)
    if key in tmdb_cache:
        return tmdb_cache[key]
    try:
        url = f'{TMDB_BASE}/movie/{tmdb_id}'
        r   = requests.get(url, params={'api_key': api_key}, timeout=8)
        if r.status_code == 200:
            d = r.json()
            result = {
                'tmdbId'       : tmdb_id,
                'budget'       : d.get('budget', 0),
                'revenue'      : d.get('revenue', 0),
                'runtime'      : d.get('runtime') or 0,
                'vote_average' : d.get('vote_average', 0.0),
                'vote_count'   : d.get('vote_count', 0),
                'popularity'   : d.get('popularity', 0.0),
            }
            tmdb_cache[key] = result
            return result
        elif r.status_code == 429:          # Rate limit
            time.sleep(2)
            return fetch_tmdb_metadata(tmdb_id, api_key)
    except Exception:
        pass
    return None

Loaded 32,194 entries from cache


In [13]:
# Fetch metadata for all movies with tmdbId
links_with_tmdb = df_links.dropna(subset=['tmdbId']).copy()
total           = len(links_with_tmdb)
print(f'Fetching TMDB metadata for {total:,} movies...')
print('Estimated time: ~{:.0f} min at 4 req/s'.format(total / 4 / 60))

records = []
save_every = 500   # Save cache every 500 requests

for i, (_, row) in enumerate(tqdm(links_with_tmdb.iterrows(), total=total)):
    meta = fetch_tmdb_metadata(int(row['tmdbId']))
    if meta:
        meta['movieId'] = int(row['movieId'])
        records.append(meta)
    time.sleep(0.25)   # 4 req/s

    # Save cache periodically
    if (i + 1) % save_every == 0:
        with open(CACHE_FILE, 'w') as f:
            json.dump(tmdb_cache, f)

# Final cache save
with open(CACHE_FILE, 'w') as f:
    json.dump(tmdb_cache, f)

df_tmdb = pd.DataFrame(records)
print(f'\nGot metadata for {len(df_tmdb):,} / {total:,} movies')
display(df_tmdb.head(3))

Fetching TMDB metadata for 32,700 movies...
Estimated time: ~136 min at 4 req/s


  0%|          | 0/32700 [00:00<?, ?it/s]


Got metadata for 32,224 / 32,700 movies


,tmdbId,budget,revenue,runtime,vote_average,vote_count,popularity,movieId
0,862,30000000,401157969,81,7.973,19847,24.4839,1
1,8844,65000000,262821940,104,7.246,11287,2.8059,2
2,15602,25000000,71518503,101,6.479,428,1.8211,3


In [14]:
# -- Compute IMDb Score (Bayesian Weighted Rating) --------------------------
# Filter movies with vote_count >= 10 for a reliable imdb_score
df_tmdb = df_tmdb[df_tmdb['vote_count'] >= 10].copy()

C = df_tmdb['vote_average'].mean()           # Global mean
m = df_tmdb['vote_count'].quantile(0.60)     # 60th percentile as minimum threshold
v = df_tmdb['vote_count']
R = df_tmdb['vote_average']

df_tmdb['imdb_score'] = (v / (v + m)) * R + (m / (v + m)) * C

print(f'C (mean vote_average) = {C:.3f}')
print(f'm (60th percentile vote_count) = {m:.0f}')
print(f'imdb_score range: {df_tmdb["imdb_score"].min():.3f} - {df_tmdb["imdb_score"].max():.3f}')
print(f'imdb_score mean:  {df_tmdb["imdb_score"].mean():.3f}')
display(df_tmdb[['movieId', 'vote_count', 'vote_average', 'imdb_score']].sort_values('imdb_score', ascending=False).head(10))

C (mean vote_average) = 6.289
m (60th percentile vote_count) = 157
imdb_score range: 3.126 - 8.707
imdb_score mean:  6.359


,movieId,vote_count,vote_average,imdb_score
314,318,30368,8.720,8.707497
837,858,22916,8.686,8.669690
522,527,17433,8.569,8.548651
1187,1221,13894,8.571,8.545503
1170,1203,9971,8.561,8.525782
11974,58559,35739,8.530,8.520199
5481,5618,18295,8.534,8.514899
3044,3147,19231,8.506,8.488048
6982,7153,26500,8.497,8.483996
31994,202439,20603,8.494,8.477325


## IV. Genome Tags — Top-3 Tags per Movie

`genome-scores.csv` contains relevance scores for 1128 tags x ~13K movies.  
Extract the top-3 highest-relevance tags per movie — used in Content-Based Filtering.

In [15]:
# Filter genome scores to valid movies only
df_genome_scores = df_genome_scores[df_genome_scores['movieId'].isin(valid_movies)]

# Map tagId to tag string
tag_id_to_name = df_genome_tags.set_index('tagId')['tag'].to_dict()

# Top-3 genome tags per movie
df_genome_top3 = (
    df_genome_scores
    .sort_values(['movieId', 'relevance'], ascending=[True, False])
    .groupby('movieId')
    .head(3)
    .copy()
)
df_genome_top3['tag_name'] = df_genome_top3['tagId'].map(tag_id_to_name)

df_genome_agg = (
    df_genome_top3
    .groupby('movieId')['tag_name']
    .apply(lambda x: ' '.join(x.dropna()))
    .reset_index()
    .rename(columns={'tag_name': 'genome_tags'})
)
print(f'Movies with genome tags: {len(df_genome_agg):,}')
display(df_genome_agg.head(5))

Movies with genome tags: 13,816


,movieId,genome_tags
0,1,toys computer animation pixar animation
1,2,adventure children fantasy
2,3,sequel good sequel sequels
3,4,women chick flick divorce
4,5,sequel good sequel father daughter relationship


## V. Merge & Feature Selection

In [16]:
# Merge all tables
df_merge = (
    df_movies
    .merge(df_links[['movieId', 'tmdbId', 'imdbId']], on='movieId', how='left')
    .merge(df_tmdb[['movieId', 'budget', 'revenue', 'runtime',
                    'vote_average', 'vote_count', 'popularity', 'imdb_score']],
           on='movieId', how='left')
    .merge(df_tags_agg,     on='movieId', how='left')
    .merge(df_genome_agg,   on='movieId', how='left')
)

# Handle nulls after merge
df_merge['tag']         = df_merge['tag'].fillna('')
df_merge['genome_tags'] = df_merge['genome_tags'].fillna('')
for col in ['vote_count', 'vote_average', 'popularity', 'runtime']:
    df_merge[col] = df_merge[col].fillna(df_merge[col].median())
df_merge['budget']    = df_merge['budget'].fillna(0)
df_merge['revenue']   = df_merge['revenue'].fillna(0)
df_merge['imdb_score'] = df_merge['imdb_score'].fillna(df_merge['imdb_score'].median())

print('Merged shape:', df_merge.shape)
display(df_merge.head(3))

Merged shape: (32743, 14)


,movieId,title,genres,tmdbId,imdbId,budget,revenue,runtime,vote_average,vote_count,popularity,imdb_score,tag,genome_tags
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862,114709,30000000.0,401157969.0,81.0,7.973,19847.0,24.4839,7.959784,owned imdb top 250 pixar time travel children ...,toys computer animation pixar animation
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844,113497,65000000.0,262821940.0,104.0,7.246,11287.0,2.8059,7.232872,robin williams time travel fantasy based on ch...,adventure children fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,15602,113228,25000000.0,71518503.0,101.0,6.479,428.0,1.8211,6.428033,funny best friend duringcreditsstinger fishing...,sequel good sequel sequels


In [17]:
# Feature selection - keep necessary columns
keep_cols = [
    'movieId', 'title', 'genres',
    'popularity', 'vote_count', 'vote_average',
    'budget', 'revenue', 'runtime',
    'tag', 'genome_tags', 'imdb_score'
]
df_reduced = df_merge[keep_cols].copy()
print('Reduced shape:', df_reduced.shape)

Reduced shape: (32743, 12)


## VI. Data Transformation

In [18]:
df_norm = df_reduced.copy()

# 1. Genres: string to list
df_norm['genres'] = df_norm['genres'].fillna('').str.split('|')

# 2. Compute avg_rating per movie from ratings table
avg_rat = df_ratings.groupby('movieId')['rating'].mean().rename('avg_rating').reset_index()
df_norm = df_norm.merge(avg_rat, on='movieId', how='left')
df_norm['avg_rating'] = df_norm['avg_rating'].fillna(df_norm['avg_rating'].median())

# 3. Min-Max Normalization for numeric columns
scale_cols = ['popularity', 'vote_count', 'budget', 'revenue',
              'runtime', 'avg_rating', 'vote_average', 'imdb_score']

# Clip negative values before log transform
for col in scale_cols:
    df_norm[col] = df_norm[col].clip(lower=0)
    # Log1p to reduce skewness
    if col in ['budget', 'revenue', 'popularity', 'vote_count']:
        df_norm[col] = np.log1p(df_norm[col])

scaler = MinMaxScaler()
df_norm[scale_cols] = scaler.fit_transform(df_norm[scale_cols])
print('Normalization done')

# 4. OneHot Encoding genres
all_genres = sorted(set(g for row in df_norm['genres'] for g in row if g and g != '(no genres listed)'))
print(f'Unique genres: {len(all_genres)}')
for g in all_genres:
    safe_col = 'genre_' + g.replace('-', '_').replace(' ', '_').replace("'", '')
    df_norm[safe_col] = df_norm['genres'].apply(lambda lst: 1 if g in lst else 0)

Normalization done
Unique genres: 19


In [19]:
# 5. TF-IDF for user-generated tags
df_norm['tag'] = df_norm['tag'].fillna('').str.lower().str.strip()
tfidf_tag = TfidfVectorizer(max_features=100, stop_words='english', min_df=5)
tag_matrix = tfidf_tag.fit_transform(df_norm['tag'])
tag_df = pd.DataFrame(tag_matrix.toarray(),
                      columns=['tag_' + c for c in tfidf_tag.get_feature_names_out()],
                      index=df_norm.index)
print(f'TF-IDF tags shape: {tag_df.shape}')

# 6. TF-IDF for genome tags (ml-25m exclusive)
df_norm['genome_tags'] = df_norm['genome_tags'].fillna('').str.lower().str.strip()
tfidf_genome = TfidfVectorizer(max_features=50, stop_words='english', min_df=2)
genome_matrix = tfidf_genome.fit_transform(df_norm['genome_tags'])
genome_df = pd.DataFrame(genome_matrix.toarray(),
                         columns=['genome_' + c for c in tfidf_genome.get_feature_names_out()],
                         index=df_norm.index)
print(f'Genome TF-IDF shape: {genome_df.shape}')

# Combine all feature columns
df_features = pd.concat([df_norm, tag_df, genome_df], axis=1)
print(f'Total features shape: {df_features.shape}')

TF-IDF tags shape: (32743, 100)
Genome TF-IDF shape: (32743, 50)
Total features shape: (32743, 182)


## VII. Data Discretization

In [20]:
# Extract release year from title
def extract_year(title):
    m = re.search(r'\((\d{4})\)', str(title))
    return int(m.group(1)) if m else 2000

df_features['year'] = df_features['title'].apply(extract_year)

# Categorize movie era
df_features['movie_era'] = pd.cut(
    df_features['year'],
    bins=[0, 1970, 1990, 2000, 2010, 2020, 9999],
    labels=['Classic', '70s-80s', '90s', '2000s', '2010s', 'Recent']
)

# Categorize runtime (column is already normalized, use raw values for qcut)
runtime_raw = df_reduced['runtime'].clip(lower=1)
df_features['runtime_category'] = pd.qcut(
    runtime_raw, q=3, labels=['Short', 'Medium', 'Long']
)

print('Discretization done')
print(df_features[['title', 'year', 'movie_era', 'runtime_category']].head(8))

Discretization done
                                title  year movie_era runtime_category
0                    Toy Story (1995)  1995       90s            Short
1                      Jumanji (1995)  1995       90s           Medium
2             Grumpier Old Men (1995)  1995       90s           Medium
3            Waiting to Exhale (1995)  1995       90s             Long
4  Father of the Bride Part II (1995)  1995       90s             Long
5                         Heat (1995)  1995       90s             Long
6                      Sabrina (1995)  1995       90s             Long
7                 Tom and Huck (1995)  1995       90s           Medium


## VIII. Export Data

In [21]:
def safe_save(df, name):
    """Save DataFrame as CSV and Parquet."""
    # Convert non-serializable dtypes
    df_save = df.copy()
    for col in df_save.select_dtypes(include='category').columns:
        df_save[col] = df_save[col].astype(str)
    for col in df_save.columns:
        if df_save[col].dtype == object:
            try:
                df_save[col] = df_save[col].apply(
                    lambda x: '|'.join(x) if isinstance(x, list) else x
                )
            except Exception:
                pass

    csv_path = DATA_CLEAN / f'{name}.csv'
    pq_path  = DATA_CLEAN / f'{name}.parquet'
    df_save.to_csv(csv_path, index=False, encoding='utf-8')
    df_save.to_parquet(pq_path, index=False)
    print(f'Saved {name}: {len(df_save):,} rows | '
          f'CSV={csv_path.stat().st_size/1e6:.1f}MB | '
          f'Parquet={pq_path.stat().st_size/1e6:.1f}MB')

# Clean ratings
df_ratings_save = df_ratings[['userId', 'movieId', 'rating', 'timestamp']]
safe_save(df_ratings_save, 'ratings_cleaning')

# Full movie features (for Content-Based)
safe_save(df_features, 'model_features')

# Compact movie metadata (for app and testing)
meta_cols = ['movieId', 'title', 'genres', 'imdb_score', 'vote_average', 'popularity', 'movie_era', 'runtime_category']
df_movies_meta = df_features[[c for c in meta_cols if c in df_features.columns]].drop_duplicates('movieId')
safe_save(df_movies_meta, 'movies_metadata')

# Clean links
safe_save(df_links[['movieId', 'imdbId', 'tmdbId']], 'links_cleaning')

print('\nData Processing complete.')
print(f'   Ratings : {len(df_ratings_save):,}')
print(f'   Movies  : {len(df_movies_meta):,}')
print(f'   Features: {df_features.shape[1]} columns')

Saved ratings_cleaning: 24,945,870 rows | CSV=676.7MB | Parquet=165.5MB
Saved model_features: 32,743 rows | CSV=34.7MB | Parquet=7.0MB
Saved movies_metadata: 32,720 rows | CSV=3.9MB | Parquet=1.5MB
Saved links_cleaning: 32,720 rows | CSV=0.7MB | Parquet=0.6MB

Data Processing complete.
   Ratings : 24,945,870
   Movies  : 32,720
   Features: 185 columns


## IX. Processing Pipeline Summary

| Step | Input | Output | Notes |
|------|-------|--------|---------|
| Load | ml-25m (6 files) | 6 DataFrames | |
| Cleaning | Raw DFs | Filtered DFs | User/Movie >= 5 ratings |
| TMDB API | links.csv -> tmdbId | df_tmdb | Resumable via cache |
| IMDb Score | vote_avg + vote_cnt | imdb_score | Bayesian Weighted Rating |
| Genome Tags | genome-scores.csv | genome_tags col | Top-3 tags per movie |
| Merge | 5 DFs | df_merge | left join on movieId |
| Feature Eng. | text + numeric | OneHot + TF-IDF | 100 tag + 50 genome feats |
| Discretization | year, runtime | movie_era, runtime_cat | 6 era bins, 3 runtime bins |
| Export | df_features | CSV + Parquet | Parquet 5-10x smaller than CSV |